# 05 - Atualizacao do Plenario da Camara

Atualiza somente a janela incremental de `2026-05-01` a `2026-07-13`. O run historico iniciado em 1946 fica preservado no Drive, mas fora dos gates deste ciclo.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
ACTIVE_CONFIG_PATH = DATA_ROOT / "operations" / "atualizacao" / "active.json"
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # Opcional: branch, tag ou commit. Vazio usa o default remoto.

os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
for name in ["raw", "checkpoints", "logs", "manifests", "processed", "operations/atualizacao"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("DATA_ROOT:", DATA_ROOT)
print("Repositorio:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

In [ ]:
EXPECTED_CYCLE_ID = "20260713"
if not ACTIVE_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Controle ativo ausente: {ACTIVE_CONFIG_PATH}. Execute o caderno 00 primeiro.")
CONFIG = json.loads(ACTIVE_CONFIG_PATH.read_text(encoding="utf-8"))
assert CONFIG["schema_version"] == 1
assert CONFIG["cycle_id"] == EXPECTED_CYCLE_ID, CONFIG["cycle_id"]
assert CONFIG["window"] == {"data_inicio": "2026-05-01", "data_fim": "2026-07-13"}
assert CONFIG["data_inicio"] == CONFIG["window"]["data_inicio"]
assert CONFIG["data_fim"] == CONFIG["window"]["data_fim"]
assert Path(CONFIG["data_root"]) == DATA_ROOT
ALL_CONFIGURED_RUNS = {item["key"]: item for item in CONFIG["collection_runs"]}
PRESERVED_OUT_OF_SCOPE_RUNS = {
    item["key"]: item for item in CONFIG.get("preserved_out_of_scope_runs", [])
}
CAMARA_PLENARIO_HISTORICAL_SCOPE = {
    "key": "camara_plenario_historico",
    "source": "camara",
    "dataset": "plenario_discursos",
    "run_id": "prod-historico-camara-plenario",
    "data_inicio": "1946-01-01",
    "data_fim": "2026-05-28",
}
historical_candidate = (
    ALL_CONFIGURED_RUNS.get("camara_plenario_historico")
    or PRESERVED_OUT_OF_SCOPE_RUNS.get("camara_plenario_historico")
)
SCOPE_EXCLUDED_RUNS = {}
if historical_candidate is not None:
    for field, expected in CAMARA_PLENARIO_HISTORICAL_SCOPE.items():
        assert historical_candidate.get(field) == expected, {
            "field": field,
            "expected": expected,
            "actual": historical_candidate.get(field),
        }
    SCOPE_EXCLUDED_RUNS[historical_candidate["key"]] = historical_candidate
RUNS = {
    key: run
    for key, run in ALL_CONFIGURED_RUNS.items()
    if key not in SCOPE_EXCLUDED_RUNS
}
SCOPE_EXCLUSION_RESULTS = {
    key: {
        "run_id": run["run_id"],
        "status": "out_of_scope",
        "reason": (
            "O ciclo 20260713 e uma atualizacao incremental de no maximo tres meses; "
            "a recuperacao iniciada em 1946 nao e requisito desta atualizacao."
        ),
        "preserved_artifacts": ["raw", "checkpoint", "log", "autosave", "manifest"],
    }
    for key, run in SCOPE_EXCLUDED_RUNS.items()
}
print("Ciclo ativo:", CONFIG["cycle_id"], CONFIG["window"])
print("Coletas fora do escopo incremental:", SCOPE_EXCLUSION_RESULTS)

In [ ]:
from contextlib import contextmanager
from datetime import datetime, timezone

TERMINAL_STATUSES = {"completed"}
DEFERRED_COLLECTIONS_PATH = (
    DATA_ROOT
    / "operations"
    / "atualizacao"
    / "ciclos"
    / EXPECTED_CYCLE_ID
    / "deferred_collections.json"
)
DEFERRED_COLLECTION_POLICIES = {
    "senado_ccj_historico": {
        "run_id": "prod-historico-senado-ccj",
        "allowed_statuses": ["completed_with_errors"],
        "allowed_unresolved_partitions": ["2015-05"],
        "analysis_exclusion": "senado/ccj_notas",
    }
}

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    content = path.read_text(encoding="utf-8")
    return json.loads(content) if content.strip() else None

def manifest_for(run):
    return DATA_ROOT / "manifests" / f"{run['run_id']}.json"

def checkpoint_for(run):
    return DATA_ROOT / "checkpoints" / run["source"] / f"{run['dataset']}.json"

def unresolved_partitions(run):
    checkpoint = read_json(checkpoint_for(run)) or {}
    current = (checkpoint.get("runs") or {}).get(run["run_id"], {}) or {}
    failed = set((current.get("failed_partitions") or {}).keys())
    completed = set((current.get("completed_partitions") or {}).keys())
    return sorted(failed - completed)

def _assert_manifest_contract(run):
    manifest = read_json(manifest_for(run))
    assert manifest is not None, f"Manifest final ausente: {manifest_for(run)}"
    assert manifest.get("run_id") == run["run_id"]
    assert manifest.get("mode") == "prod", (run["key"], manifest.get("mode"))
    assert manifest.get("sample") is False, (run["key"], manifest.get("sample"))
    assert manifest.get("data_inicio") == run["data_inicio"], (run["key"], manifest.get("data_inicio"))
    assert manifest.get("data_fim") == run["data_fim"], (run["key"], manifest.get("data_fim"))
    return manifest

def assert_collection_complete(run):
    manifest = _assert_manifest_contract(run)
    assert manifest.get("status") in TERMINAL_STATUSES, (run["key"], manifest.get("status"))
    unresolved = unresolved_partitions(run)
    assert not unresolved, f"Particoes falhas nao resolvidas em {run['key']}: {unresolved[:20]}"
    return manifest

def deferred_collection_for(run):
    payload = read_json(DEFERRED_COLLECTIONS_PATH)
    if not payload:
        return None
    assert payload.get("schema_version") == 1, DEFERRED_COLLECTIONS_PATH
    assert payload.get("cycle_id") == EXPECTED_CYCLE_ID, payload.get("cycle_id")
    for item in payload.get("items", []):
        if item.get("key") == run["key"]:
            policy = DEFERRED_COLLECTION_POLICIES.get(run["key"])
            assert policy is not None, f"Adiamento sem politica: {run['key']}"
            assert item.get("run_id") == run["run_id"] == policy["run_id"], item
            assert item.get("allowed_statuses") == policy["allowed_statuses"], item
            assert (
                item.get("allowed_unresolved_partitions")
                == policy["allowed_unresolved_partitions"]
            ), item
            assert item.get("analysis_exclusion") == policy["analysis_exclusion"], item
            return item
    return None

def collection_acceptance(run):
    try:
        manifest = assert_collection_complete(run)
        return {
            "manifest": manifest,
            "deferred": False,
            "status": manifest.get("status"),
            "unresolved": [],
        }
    except AssertionError as strict_error:
        deferral = deferred_collection_for(run)
        assert deferral is not None, strict_error
        manifest = _assert_manifest_contract(run)
        allowed_statuses = set(deferral.get("allowed_statuses") or [])
        allowed_unresolved = sorted(deferral.get("allowed_unresolved_partitions") or [])
        actual_unresolved = unresolved_partitions(run)
        assert deferral.get("analysis_excluded") is True, deferral
        assert str(deferral.get("reason") or "").strip(), deferral
        assert manifest.get("status") in allowed_statuses, (
            run["key"], manifest.get("status"), sorted(allowed_statuses)
        )
        assert actual_unresolved == allowed_unresolved, {
            "run": run["key"],
            "expected_unresolved": allowed_unresolved,
            "actual_unresolved": actual_unresolved,
        }
        return {
            "manifest": manifest,
            "deferred": True,
            "status": manifest.get("status"),
            "unresolved": actual_unresolved,
            "reason": deferral["reason"],
            "follow_up": deferral.get("follow_up"),
        }

def assert_collection_accepted(run):
    return collection_acceptance(run)["manifest"]

def show_run_state(run, tail_lines=5):
    final = read_json(manifest_for(run))
    autosave_path = DATA_ROOT / "manifests" / f"{run['run_id']}.autosave.json"
    autosave = read_json(autosave_path)
    log_path = DATA_ROOT / "logs" / f"{run['run_id']}.jsonl"
    tail = log_path.read_text(encoding="utf-8").splitlines()[-tail_lines:] if log_path.exists() else []
    print(run["key"], {
        "manifest": str(manifest_for(run)),
        "status": final.get("status") if final else None,
        "autosave_status": autosave.get("status") if autosave else None,
        "unresolved": unresolved_partitions(run),
        "log_tail": tail,
    })

def collector_command(run, *extra):
    return [
        sys.executable, "-u", "-m", run["module"],
        "--mode", "prod",
        "--output-dir", str(DATA_ROOT),
        "--data-inicio", run["data_inicio"],
        "--data-fim", run["data_fim"],
        "--run-id", run["run_id"],
        "--no-sample", "--resume", *extra,
    ]

def run_streamed(command, label):
    print(f"\n=== {label} ===", flush=True)
    print(" ".join(map(str, command)), flush=True)
    completed = subprocess.run(list(map(str, command)), check=False)
    returncode = completed.returncode
    print(f"=== retorno {returncode}: {label} ===", flush=True)
    return returncode

@contextmanager
def dataset_lock(run):
    lock_root = DATA_ROOT / "operations" / "atualizacao" / "locks"
    lock_root.mkdir(parents=True, exist_ok=True)
    lock_path = lock_root / f"{run['source']}__{run['dataset']}.json"
    payload = {
        "cycle_id": CONFIG["cycle_id"],
        "run_id": run["run_id"],
        "source": run["source"],
        "dataset": run["dataset"],
        "started_at": datetime.now(timezone.utc).isoformat(),
    }
    try:
        with lock_path.open("x", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
            handle.write("\n")
    except FileExistsError as exc:
        raise RuntimeError(f"Dataset ja bloqueado por outra sessao: {lock_path}\n{lock_path.read_text()}") from exc
    try:
        yield
    finally:
        if lock_path.exists() and read_json(lock_path) == payload:
            lock_path.unlink()

def run_collector(run, *extra):
    with dataset_lock(run):
        return run_streamed(collector_command(run, *extra), run["key"])

def require_explicit_confirmation(enabled, confirmation):
    if enabled:
        assert confirmation == EXPECTED_CYCLE_ID, "Digite o cycle_id na variavel CONFIRMAR_CICLO."

def assert_parlamentares_ready():
    run_id = CONFIG["processing_run_ids"]["parlamentares"]
    manifest_path = DATA_ROOT / "processed" / "manifests" / f"{run_id}-parlamentares.json"
    periodos_path = DATA_ROOT / "processed" / "parlamentares" / "v1" / "parquet" / "parlamentares_periodos.parquet"
    manifest = read_json(manifest_path)
    assert manifest and manifest.get("run_id") == run_id and manifest.get("dataset_version") == "v1", manifest_path
    assert periodos_path.exists(), periodos_path
    return manifest

## Recorte operacional deste ciclo

Esta atualizacao cobre pouco mais de dois meses, com sobreposicao desde a ultima
coleta de maio. `prod-historico-camara-plenario` nao sera retomado aqui: seu raw,
checkpoint, log, autosave e eventual manifest permanecem intactos para uma tarefa
historica separada. Um `try/except` nao resolveria a demora observada, pois as
requisicoes antigas estavam respondendo normalmente; o tratamento correto e nao
executar uma faixa que esta fora do escopo.

O Parquet pequeno de periodos parlamentares e copiado para o disco local efemero
do runtime. Todas as saidas da coleta incremental continuam no Drive.

In [ ]:
import shutil

assert "camara_plenario_historico" not in RUNS
historical = SCOPE_EXCLUDED_RUNS["camara_plenario_historico"]
incremental = RUNS["camara_plenario"]
assert incremental["data_inicio"] == CONFIG["window"]["data_inicio"] == "2026-05-01"
assert incremental["data_fim"] == CONFIG["window"]["data_fim"] == "2026-07-13"
assert incremental["run_id"] == "prod-atualizacao-20260713-camara-plenario"
LOCAL_RUNTIME_ROOT = Path("/content/falando_nela_runtime")

def cache_parlamentares_periodos():
    source = DATA_ROOT / "processed" / "parlamentares" / "v1" / "parquet" / "parlamentares_periodos.parquet"
    target = LOCAL_RUNTIME_ROOT / "parlamentares_periodos.parquet"
    assert source.is_file(), source
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
    assert target.is_file() and target.stat().st_size == source.stat().st_size, (source, target)
    print("Plano de mandatos copiado para o runtime local:", target, target.stat().st_size, "bytes")
    return target
print("Historico preservado e fora do escopo:", SCOPE_EXCLUSION_RESULTS[historical["key"]])
print("Faixa incremental:", incremental["data_inicio"], incremental["data_fim"], incremental["run_id"])
incremental_lock_path = (
    DATA_ROOT
    / "operations"
    / "atualizacao"
    / "locks"
    / f"{incremental['source']}__{incremental['dataset']}.json"
)
if incremental_lock_path.exists():
    print("LOCK EXISTENTE:", incremental_lock_path)
    print(incremental_lock_path.read_text(encoding="utf-8"))

Se o lock exibido pertencer a uma sessao historica que ja foi encerrada, confirme
no Colab que nao existe outro runtime ativo e remova manualmente **somente**
`operations/atualizacao/locks/camara__plenario_discursos.json`. Nao remova raw,
checkpoint, log, autosave ou manifest. Se outra sessao ainda estiver ativa, pare
aqui para evitar duas escritas simultaneas no mesmo dataset.

In [ ]:
RODAR_FAIXA_INCREMENTAL = False
CONFIRMAR_CICLO = ""
require_explicit_confirmation(RODAR_FAIXA_INCREMENTAL, CONFIRMAR_CICLO)
if RODAR_FAIXA_INCREMENTAL:
    assert_parlamentares_ready()
    local_periodos = cache_parlamentares_periodos()
    rc = run_collector(
        incremental,
        "--parlamentares-periodos-path", local_periodos,
    )
    assert rc == 0
    assert_collection_complete(incremental)
else:
    print("Incremental protegido: RODAR_FAIXA_INCREMENTAL=False")

## Estado da atualizacao e artefatos preservados

In [ ]:
show_run_state(incremental)
historical_manifest_path = manifest_for(historical)
historical_autosave_path = DATA_ROOT / "manifests" / f"{historical['run_id']}.autosave.json"
print("Historico fora do escopo:", {
    "run_id": historical["run_id"],
    "manifest_path": str(historical_manifest_path),
    "manifest_bytes": historical_manifest_path.stat().st_size if historical_manifest_path.exists() else None,
    "autosave_path": str(historical_autosave_path),
    "autosave_exists": historical_autosave_path.exists(),
    "raw_preservado": str(DATA_ROOT / "raw" / historical["source"] / historical["dataset"]),
})